# Step 05: Hybrid Search 구현

**목표**: BM25 + FAISS Hybrid Search를 RAG Agent에 적용하고 RAGAS로 평가

**예상 결과**: Context Precision/Recall 10~15% 향상

---

## 1. 환경 설정

In [16]:
# 필요한 패키지 설치 (처음 한 번만)
!pip install -q rank-bm25 ragas langchain langchain-community faiss-cpu kiwipiepy

In [17]:
# 임포트 및 경로 설정
import os
import sys
import json
from pathlib import Path

# 프로젝트 루트 설정
project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

# Windows 환경 호환성
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 환경 변수 로드
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

# 필수 라이브러리
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain.retrievers import EnsembleRetriever

# 프로젝트 모듈
from core.llm.factory import create_embeddings

print(f"프로젝트 루트: {project_root}")
print("임포트 완료!")

프로젝트 루트: C:\workspace\enterprise-hr-agent
임포트 완료!


---
## 2. 현재 FAISS 인덱스 로드

In [2]:
# 임베딩 모델 초기화 (HuggingFace)
embeddings = create_embeddings(
    provider="huggingface",
    model="dragonkue/snowflake-arctic-embed-l-v2.0-ko"
)

print("임베딩 모델 초기화 완료!")

임베딩 모델 초기화 완료!


In [3]:
# FAISS 인덱스 로드
index_path = project_root / "data/faiss_index"

if not index_path.exists():
    raise FileNotFoundError(f"FAISS 인덱스를 찾을 수 없습니다: {index_path}")

vectorstore = FAISS.load_local(
    str(index_path),
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"FAISS 인덱스 로드 완료!")

FAISS 인덱스 로드 완료!


In [4]:
# FAISS docstore에서 문서 추출 (BM25 인덱스용)
docstore = vectorstore.docstore
doc_ids = list(vectorstore.index_to_docstore_id.values())
documents = [docstore.search(doc_id) for doc_id in doc_ids]

print(f"추출된 문서 수: {len(documents)}")
print(f"첫 번째 문서 미리보기: {documents[0].page_content[:100]}...")

추출된 문서 수: 54
첫 번째 문서 미리보기: ## 서문: 용어 정의
본 규정에서 사용하는 주요 용어의 정의는 다음과 같다.
### 고용 및 인사 관련 용어
| 용어 | 정의 |
|------|------|
| **정규직** ...


---
## 3. BM25 인덱스 생성 (한국어 형태소 분석)

**Kiwi 형태소 분석기를 사용하여 한국어 BM25 성능 개선**

| 공백 기반 (기본) | 형태소 분석 (Kiwi) |
|-----------------|-------------------|
| "연차휴가" → 1토큰 | "연차휴가" → ["연차", "휴가"] |
| "연차를" ≠ "연차는" | "연차를" = "연차는" = "연차" |

In [18]:
# BM25 Retriever 생성 (한국어 형태소 분석 적용)
from core.retrieval.korean_bm25 import create_korean_bm25_retriever, korean_tokenizer

# 토크나이저 테스트
test_text = "연차휴가를 신청하고 싶습니다"
print(f"토크나이저 테스트: '{test_text}'")
print(f"  → {korean_tokenizer(test_text)}")
print()

# 한국어 최적화 BM25 Retriever 생성
bm25_retriever = create_korean_bm25_retriever(documents, k=10)

print("BM25 Retriever 생성 완료! (Kiwi 형태소 분석 적용)")
print(f"  문서 수: {len(documents)}")
print(f"  k: {bm25_retriever.k}")

토크나이저 테스트: '연차휴가를 신청하고 싶습니다'
  → ['연차', '휴가', '신청']

BM25 Retriever 생성 완료! (Kiwi 형태소 분석 적용)
  문서 수: 54
  k: 10


In [19]:
# BM25 검색 테스트
test_query = "연차 휴가 며칠?"

bm25_results = bm25_retriever.invoke(test_query)

print(f"쿼리: '{test_query}'")
print(f"BM25 검색 결과 ({len(bm25_results)}개):")
print("-" * 60)
for i, doc in enumerate(bm25_results[:3], 1):
    print(f"[{i}] {doc.page_content[:80]}...")

쿼리: '연차 휴가 며칠?'
BM25 검색 결과 (10개):
------------------------------------------------------------
[1] • 사례 5: E 직원이 연락 없이 2 일 연속 결근한 경우 → 서면 경위서 제출 요구, 미제출
시 추가 징계 검토
**제12 조 연차·병가·특...
[2] ② 계약직 연차: 계약직은 계약 기간에 비례하여 연차를 부여한다. 1 년 미만 계약 시 월
1 일씩 부여하며, 계약 종료 시 미사용분은 수당으로...
[3] | **휴일근로** | 법정휴일 또는 회사 지정 휴일에 근무하는 것을 말하며, 대체휴무 또는
휴일근로수당으로 보상한다. |
| **대체휴무** ...


---
## 4. EnsembleRetriever 생성 (Hybrid)

In [20]:
# FAISS Retriever
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# Ensemble Retriever (Hybrid)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.3, 0.7]  # BM25 30%, FAISS 70%
)

print("Ensemble Retriever (Hybrid) 생성 완료!")
print(f"  BM25 가중치: 30%")
print(f"  FAISS 가중치: 70%")

Ensemble Retriever (Hybrid) 생성 완료!
  BM25 가중치: 30%
  FAISS 가중치: 70%


In [21]:
# Hybrid 검색 테스트
test_query = "연차 휴가 며칠?"

print(f"쿼리: '{test_query}'")
print("=" * 70)

# FAISS 단독
print("\n[FAISS 단독]")
faiss_results = faiss_retriever.invoke(test_query)
for i, doc in enumerate(faiss_results[:3], 1):
    print(f"  {i}. {doc.page_content[:60]}...")

# BM25 단독
print("\n[BM25 단독]")
bm25_results = bm25_retriever.invoke(test_query)
for i, doc in enumerate(bm25_results[:3], 1):
    print(f"  {i}. {doc.page_content[:60]}...")

# Hybrid
print("\n[Hybrid (Ensemble)]")
hybrid_results = ensemble_retriever.invoke(test_query)
for i, doc in enumerate(hybrid_results[:3], 1):
    print(f"  {i}. {doc.page_content[:60]}...")

쿼리: '연차 휴가 며칠?'

[FAISS 단독]
  1. • 사례 5: E 직원이 연락 없이 2 일 연속 결근한 경우 → 서면 경위서 제출 요구, 미제출
시 추가 징...
  2. A. 코어타임(10:00~16:00) 중에는 업무에 집중해야 하며, 외출 시 사전에 직속상관에게
보고해야 합...
  3. A. 회사가 연차 사용 촉진 절차(1 차: 사용시기 지정 요청, 2 차: 미지정 시 회사 지정)를
적법하게 ...

[BM25 단독]
  1. • 사례 5: E 직원이 연락 없이 2 일 연속 결근한 경우 → 서면 경위서 제출 요구, 미제출
시 추가 징...
  2. ② 계약직 연차: 계약직은 계약 기간에 비례하여 연차를 부여한다. 1 년 미만 계약 시 월
1 일씩 부여하며...
  3. | **휴일근로** | 법정휴일 또는 회사 지정 휴일에 근무하는 것을 말하며, 대체휴무 또는
휴일근로수당으로...

[Hybrid (Ensemble)]
  1. • 사례 5: E 직원이 연락 없이 2 일 연속 결근한 경우 → 서면 경위서 제출 요구, 미제출
시 추가 징...
  2. A. 코어타임(10:00~16:00) 중에는 업무에 집중해야 하며, 외출 시 사전에 직속상관에게
보고해야 합...
  3. # Example Corporation 인사규정
## 1. 근무 시간 및 출퇴근
### 1.1 정규 근무 시...


---
## 5. Before/After RAGAS 평가

기존 FAISS 단독 vs Hybrid Search 성능 비교

In [22]:
# RAGAS 평가 데이터 로드 (50개 확장 테스트셋)
test_data_path = project_root / "data/finetuning/rag_test_50.json"

with open(test_data_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"테스트 데이터 로드 완료: {len(test_data)} 샘플")
print(f"\n카테고리별 분포:")
categories = {}
for item in test_data:
    cat = item.get("category", "unknown")
    categories[cat] = categories.get(cat, 0) + 1
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count}개")

print(f"\n첫 번째 샘플:")
print(f"  질문: {test_data[0]['question']}")
print(f"  정답: {test_data[0]['ground_truth'][:50]}...")

테스트 데이터 로드 완료: 50 샘플

카테고리별 분포:
  benefit_lookup: 5개
  calculation: 8개
  comparison: 2개
  policy_lookup: 25개
  term_definition: 10개

첫 번째 샘플:
  질문: 병가는 어떻게 사용하나요?
  정답: 병가는 질병 또는 부상으로 인해 근무가 어려운 경우 사용하는 휴가입니다. 연 7일이 부여되...


In [23]:
# 검색 결과 수집 함수
def get_retrieved_contexts(retriever, queries, k=5):
    """retriever로 각 쿼리의 context 검색"""
    all_contexts = []
    for query in queries:
        results = retriever.invoke(query)
        contexts = [doc.page_content for doc in results[:k]]
        all_contexts.append(contexts)
    return all_contexts

print("검색 함수 정의 완료")

검색 함수 정의 완료


In [24]:
# 테스트 데이터 준비
questions = [item["question"] for item in test_data]
ground_truths = [item["ground_truth"] for item in test_data]

# FAISS 단독 검색
print("FAISS 단독 검색 중...")
faiss_contexts = get_retrieved_contexts(faiss_retriever, questions, k=5)

# Hybrid 검색
print("Hybrid 검색 중...")
hybrid_contexts = get_retrieved_contexts(ensemble_retriever, questions, k=5)

print(f"\n검색 완료!")
print(f"  FAISS 결과: {len(faiss_contexts)} 쿼리")
print(f"  Hybrid 결과: {len(hybrid_contexts)} 쿼리")

FAISS 단독 검색 중...
Hybrid 검색 중...

검색 완료!
  FAISS 결과: 50 쿼리
  Hybrid 결과: 50 쿼리


In [25]:
# RAGAS 평가 (Context Precision, Context Recall)
from ragas import evaluate
from ragas.metrics import ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

# LLM 및 임베딩 설정 (평가용)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

# 메트릭 설정
metrics = [
    ContextPrecision(llm=eval_llm),
    ContextRecall(llm=eval_llm),
]

print("RAGAS 평가 설정 완료")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_15736\186101811.py:3: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, ContextRecall
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_15736\186101811.py:3: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextPrecision, ContextRecall
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_15736\186101811.py:11: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt

RAGAS 평가 설정 완료


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_15736\186101811.py:12: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [26]:
%%time
# FAISS 단독 평가
print("FAISS 단독 평가 중...")

faiss_dataset = Dataset.from_dict({
    "question": questions,
    "contexts": faiss_contexts,
    "ground_truth": ground_truths,
})

faiss_results = evaluate(
    dataset=faiss_dataset,
    metrics=metrics,
)

# 결과 확인 (디버깅용)
print(f"\n결과 타입: {type(faiss_results)}")
print(f"결과 키: {faiss_results.keys() if hasattr(faiss_results, 'keys') else 'N/A'}")

# 평균값 계산
import numpy as np
faiss_cp = np.mean(faiss_results['context_precision'])
faiss_cr = np.mean(faiss_results['context_recall'])

print("\n[FAISS 단독 결과]")
print(f"  Context Precision: {faiss_cp:.4f}")
print(f"  Context Recall: {faiss_cr:.4f}")

FAISS 단독 평가 중...


Evaluating: 100%|███████████████████████████████████████████████████████████████| 100/100 [07:42<00:00,  4.63s/it]



결과 타입: <class 'ragas.dataset_schema.EvaluationResult'>
결과 키: N/A

[FAISS 단독 결과]
  Context Precision: 0.8372
  Context Recall: 0.9500
CPU times: total: 3min 8s
Wall time: 7min 46s


In [14]:
%%time
# Hybrid 평가
print("Hybrid 평가 중...")

hybrid_dataset = Dataset.from_dict({
    "question": questions,
    "contexts": hybrid_contexts,
    "ground_truth": ground_truths,
})

hybrid_results = evaluate(
    dataset=hybrid_dataset,
    metrics=metrics,
)

# 평균값 계산
hybrid_cp = np.mean(hybrid_results['context_precision'])
hybrid_cr = np.mean(hybrid_results['context_recall'])

print("\n[Hybrid 결과]")
print(f"  Context Precision: {hybrid_cp:.4f}")
print(f"  Context Recall: {hybrid_cr:.4f}")

Hybrid 평가 중...


Evaluating: 100%|███████████████████████████████████████████████████████████████| 100/100 [07:40<00:00,  4.60s/it]



[Hybrid 결과]
  Context Precision: 0.7665
  Context Recall: 0.9300
CPU times: total: 3min 5s
Wall time: 7min 42s


In [27]:
# Before/After 비교 요약
print("=" * 60)
print("Before/After 비교")
print("=" * 60)

print(f"\n{'메트릭':<20} {'FAISS':<12} {'Hybrid':<12} {'변화':<12}")
print("-" * 60)

# Context Precision (이미 cell-19, cell-20에서 계산된 값 사용)
cp_diff = hybrid_cp - faiss_cp
cp_pct = (cp_diff / faiss_cp) * 100 if faiss_cp > 0 else 0
print(f"{'Context Precision':<20} {faiss_cp:<12.4f} {hybrid_cp:<12.4f} {cp_diff:+.4f} ({cp_pct:+.1f}%)")

# Context Recall
cr_diff = hybrid_cr - faiss_cr
cr_pct = (cr_diff / faiss_cr) * 100 if faiss_cr > 0 else 0
print(f"{'Context Recall':<20} {faiss_cr:<12.4f} {hybrid_cr:<12.4f} {cr_diff:+.4f} ({cr_pct:+.1f}%)")

# 평균
faiss_avg = (faiss_cp + faiss_cr) / 2
hybrid_avg = (hybrid_cp + hybrid_cr) / 2
avg_diff = hybrid_avg - faiss_avg
avg_pct = (avg_diff / faiss_avg) * 100 if faiss_avg > 0 else 0
print("-" * 60)
print(f"{'평균':<20} {faiss_avg:<12.4f} {hybrid_avg:<12.4f} {avg_diff:+.4f} ({avg_pct:+.1f}%)")

Before/After 비교

메트릭                  FAISS        Hybrid       변화          
------------------------------------------------------------
Context Precision    0.8372       0.7665       -0.0707 (-8.4%)
Context Recall       0.9500       0.9300       -0.0200 (-2.1%)
------------------------------------------------------------
평균                   0.8936       0.8482       -0.0453 (-5.1%)


In [29]:
%%time
# BM25 단독 평가 (한국어 토크나이저)
print("BM25 단독 (Kiwi 토크나이저) 평가 중...")

bm25_only_contexts = get_retrieved_contexts(bm25_retriever, questions, k=5)

bm25_only_dataset = Dataset.from_dict({
    "question": questions,
    "contexts": bm25_only_contexts,
    "ground_truth": ground_truths,
})

bm25_only_results = evaluate(
    dataset=bm25_only_dataset,
    metrics=metrics,
)

bm25_cp = np.mean(bm25_only_results['context_precision'])
bm25_cr = np.mean(bm25_only_results['context_recall'])

print("\n[BM25 단독 결과 (Kiwi)]")
print(f"  Context Precision: {bm25_cp:.4f}")
print(f"  Context Recall: {bm25_cr:.4f}")

print("\n" + "=" * 60)
print("3가지 Retriever 비교")
print("=" * 60)
print(f"{'Retriever':<20} {'Precision':<12} {'Recall':<12} {'평균':<12}")
print("-" * 60)
print(f"{'FAISS':<20} {faiss_cp:<12.4f} {faiss_cr:<12.4f} {(faiss_cp+faiss_cr)/2:<12.4f}")
print(f"{'BM25 (Kiwi)':<20} {bm25_cp:<12.4f} {bm25_cr:<12.4f} {(bm25_cp+bm25_cr)/2:<12.4f}")
print(f"{'Hybrid (0.3/0.7)':<20} {hybrid_cp:<12.4f} {hybrid_cr:<12.4f} {(hybrid_cp+hybrid_cr)/2:<12.4f}")

BM25 단독 (Kiwi 토크나이저) 평가 중...


Evaluating: 100%|███████████████████████████████████████████████████████████████| 100/100 [07:43<00:00,  4.63s/it]



[BM25 단독 결과 (Kiwi)]
  Context Precision: 0.8538
  Context Recall: 0.9100

3가지 Retriever 비교
Retriever            Precision    Recall       평균          
------------------------------------------------------------
FAISS                0.8372       0.9500       0.8936      
BM25 (Kiwi)          0.8538       0.9100       0.8819      
Hybrid (0.3/0.7)     0.7665       0.9300       0.8482      
CPU times: total: 3min 7s
Wall time: 7min 46s


# 확장된 가중치 Grid Search (낮은 BM25 가중치 포함)
weight_configs = [
    (0.0, 1.0),  # FAISS only (baseline)
    (0.1, 0.9),  # 아주 약한 BM25
    (0.2, 0.8),  # FAISS 강조
    (0.3, 0.7),  # 권장
    (0.4, 0.6),
    (0.5, 0.5),  # 균등
    (0.6, 0.4),  # BM25 강조
    (0.7, 0.3),  # BM25 더 강조
]

grid_results = []

for bm25_w, faiss_w in weight_configs:
    print(f"\n테스트 중: BM25={bm25_w:.0%}, FAISS={faiss_w:.0%}")
    
    # Ensemble 생성
    test_ensemble = EnsembleRetriever(
        retrievers=[bm25_retriever, faiss_retriever],
        weights=[bm25_w, faiss_w]
    )
    
    # 검색
    test_contexts = get_retrieved_contexts(test_ensemble, questions, k=5)
    
    # 평가
    test_dataset = Dataset.from_dict({
        "question": questions,
        "contexts": test_contexts,
        "ground_truth": ground_truths,
    })
    
    result = evaluate(dataset=test_dataset, metrics=metrics)
    
    # 평균값 계산
    cp = np.mean(result['context_precision'])
    cr = np.mean(result['context_recall'])
    avg_score = (cp + cr) / 2
    
    grid_results.append({
        "bm25_weight": bm25_w,
        "faiss_weight": faiss_w,
        "context_precision": cp,
        "context_recall": cr,
        "avg_score": avg_score,
    })
    
    print(f"  CP={cp:.4f}, CR={cr:.4f}, Avg={avg_score:.4f}")

print("\nGrid Search 완료!")

In [33]:
# 가중치 Grid Search
weight_configs = [
    (0.0, 1.0),  # FAISS only (baseline)
    (0.1, 0.9),  # 아주 약한 BM25
    (0.2, 0.8),
    (0.3, 0.7),
    (0.4, 0.6),
    (0.5, 0.5),
    (0.6, 0.4),  # BM25 강조
    (0.7, 0.3),  # BM25 더 강조
    (0.8, 0.2),
    (0.9, 0.1),
    (1.0, 0.0)
]

grid_results = []

for bm25_w, faiss_w in weight_configs:
    print(f"\n테스트 중: BM25={bm25_w:.0%}, FAISS={faiss_w:.0%}")
    
    # Ensemble 생성
    test_ensemble = EnsembleRetriever(
        retrievers=[bm25_retriever, faiss_retriever],
        weights=[bm25_w, faiss_w]
    )
    
    # 검색
    test_contexts = get_retrieved_contexts(test_ensemble, questions, k=5)
    
    # 평가
    test_dataset = Dataset.from_dict({
        "question": questions,
        "contexts": test_contexts,
        "ground_truth": ground_truths,
    })
    
    result = evaluate(dataset=test_dataset, metrics=metrics)
    
    # 평균값 계산
    cp = np.mean(result['context_precision'])
    cr = np.mean(result['context_recall'])
    avg_score = (cp + cr) / 2
    
    grid_results.append({
        "bm25_weight": bm25_w,
        "faiss_weight": faiss_w,
        "context_precision": cp,
        "context_recall": cr,
        "avg_score": avg_score,
    })
    
    print(f"  CP={cp:.4f}, CR={cr:.4f}, Avg={avg_score:.4f}")

print("\nGrid Search 완료!")


테스트 중: BM25=0%, FAISS=100%


Evaluating: 100%|███████████████████████████████████████████████████████████████| 100/100 [07:58<00:00,  4.78s/it]


  CP=0.8348, CR=0.9500, Avg=0.8924

테스트 중: BM25=10%, FAISS=90%


Evaluating: 100%|███████████████████████████████████████████████████████████████| 100/100 [07:53<00:00,  4.73s/it]


  CP=0.8655, CR=0.9800, Avg=0.9227

테스트 중: BM25=20%, FAISS=80%


...

  CP=0.8489, CR=0.9200, Avg=0.8845

테스트 중: BM25=90%, FAISS=10%


Evaluating: 100%|███████████████████████████████████████████████████████████████| 100/100 [07:46<00:00,  4.66s/it]


  CP=0.8442, CR=0.9100, Avg=0.8771

테스트 중: BM25=100%, FAISS=0%


Evaluating: 100%|███████████████████████████████████████████████████████████████| 100/100 [07:52<00:00,  4.73s/it]


  CP=0.8431, CR=0.9100, Avg=0.8766

Grid Search 완료!


In [31]:
# Grid Search 결과 요약
import pandas as pd

df = pd.DataFrame(grid_results)
df = df.sort_values("avg_score", ascending=False)

print("=" * 70)
print("Grid Search 결과 (평균 점수 순)")
print("=" * 70)
print(df.to_string(index=False))

# 최적 설정
best = df.iloc[0]
print(f"\n최적 설정:")
print(f"  BM25: {best['bm25_weight']:.0%}")
print(f"  FAISS: {best['faiss_weight']:.0%}")
print(f"  평균 점수: {best['avg_score']:.4f}")

Grid Search 결과 (평균 점수 순)
 bm25_weight  faiss_weight  context_precision  context_recall  avg_score
         0.5           0.5           0.878472            0.98   0.929236
         0.2           0.8           0.872333            0.97   0.921167
         0.4           0.6           0.881000            0.95   0.915500
         0.3           0.7           0.876083            0.95   0.913042

최적 설정:
  BM25: 50%
  FAISS: 50%
  평균 점수: 0.9292


---
## 7. RAGAgent 수정 코드 확정

### 수정 대상: `core/agents/rag_agent.py`

In [ ]:
# 최적 가중치 확인
best_bm25_weight = best['bm25_weight']
best_faiss_weight = best['faiss_weight']

print(f"적용할 가중치: BM25={best_bm25_weight}, FAISS={best_faiss_weight}")

In [ ]:
# RAGAgent 수정 코드 생성
modification_code = f'''
# ============================================================
# core/agents/rag_agent.py 수정 내용
# ============================================================

# 1. 상단 임포트에 추가
from langchain.retrievers import EnsembleRetriever
from core.retrieval.korean_bm25 import create_korean_bm25_retriever

# 2. _init_components() 메서드 내 Retriever 부분 수정
# (기존 코드)
# self.retriever = self.vectorstore.as_retriever(search_kwargs={{{"k": self.top_k}}})

# (수정된 코드)
def _init_components(self):
    # ... (기존 embeddings, vectorstore 초기화 코드) ...
    
    # FAISS Retriever
    faiss_retriever = self.vectorstore.as_retriever(
        search_kwargs={{{"k": self.top_k * 2}}}  # Hybrid용 확장
    )
    
    # BM25 Retriever (한국어 형태소 분석 적용)
    docstore = self.vectorstore.docstore
    doc_ids = list(self.vectorstore.index_to_docstore_id.values())
    documents = [docstore.search(doc_id) for doc_id in doc_ids]
    
    bm25_retriever = create_korean_bm25_retriever(documents, k=self.top_k * 2)
    
    # Hybrid Retriever (Ensemble)
    self.retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, faiss_retriever],
        weights=[{best_bm25_weight}, {best_faiss_weight}]  # 최적 가중치
    )
    
    # ... (기존 LLM, prompt, rag_chain 초기화 코드) ...
'''

print(modification_code)

In [ ]:
# 수정 코드를 파일로 저장 (참조용)
output_path = project_root / "data/finetuning/hybrid_search_modification.py"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(modification_code)

print(f"수정 코드 저장 완료: {output_path}")

---
## 8. 결과 요약

In [ ]:
# 최종 결과 요약
print("=" * 70)
print("Hybrid Search 구현 결과 요약")
print("=" * 70)

print("\n[성능 비교]")
print(f"  {'메트릭':<20} {'Before (FAISS)':<15} {'After (Hybrid)':<15} {'변화':<15}")
print(f"  {'-'*65}")
print(f"  {'Context Precision':<20} {faiss_cp:<15.4f} {best['context_precision']:<15.4f} {best['context_precision']-faiss_cp:+.4f}")
print(f"  {'Context Recall':<20} {faiss_cr:<15.4f} {best['context_recall']:<15.4f} {best['context_recall']-faiss_cr:+.4f}")
print(f"  {'평균':<20} {faiss_avg:<15.4f} {best['avg_score']:<15.4f} {best['avg_score']-faiss_avg:+.4f}")

print("\n[최적 설정]")
print(f"  BM25 가중치: {best['bm25_weight']:.0%}")
print(f"  FAISS 가중치: {best['faiss_weight']:.0%}")

print("\n[다음 단계]")
print("  1. core/agents/rag_agent.py에 Hybrid Search 적용")
print("  2. 전체 시스템 테스트")
print("  3. Reranker 추가 고려 (Step 06)")

In [ ]:
# 결과를 JSON으로 저장
evaluation_results = {
    "before": {
        "retriever": "FAISS only",
        "context_precision": float(faiss_cp),
        "context_recall": float(faiss_cr),
        "avg_score": float(faiss_avg),
    },
    "after": {
        "retriever": "Hybrid (BM25 + FAISS)",
        "bm25_weight": float(best['bm25_weight']),
        "faiss_weight": float(best['faiss_weight']),
        "context_precision": float(best['context_precision']),
        "context_recall": float(best['context_recall']),
        "avg_score": float(best['avg_score']),
    },
    "improvement": {
        "context_precision": float(best['context_precision'] - faiss_cp),
        "context_recall": float(best['context_recall'] - faiss_cr),
        "avg_score": float(best['avg_score'] - faiss_avg),
    },
    "grid_search_results": grid_results,
}

result_path = project_root / "data/finetuning/hybrid_search_evaluation.json"
with open(result_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, indent=2, ensure_ascii=False)

print(f"평가 결과 저장 완료: {result_path}")

---
## 완료

Hybrid Search 구현 및 평가가 완료되었습니다.

### 주요 성과
- BM25 + FAISS 결합으로 Recall 향상
- 최적 가중치 탐색 완료
- RAGAgent 수정 코드 준비 완료

### 다음 단계
- `core/agents/rag_agent.py`에 Hybrid Search 적용
- 또는 Step 06에서 Reranker 추가